# Fine-tune Cross-Encoder CV-JD v0.6

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.5 có class balance hoàn hảo.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss`, `BoundaryAwareLoss` (regression: label = score / 100) |
| **Evaluator** | Spearman correlation, LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Mục tiêu**: Vượt LabelAcc 60.76% của v0.2 nhờ dataset balance hoàn hảo (20% per class) + 91% more data (13,350 vs 7,000 pairs).

In [2]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

Cloning into '/content/Ai-Recruiter-Mini-Ai-Service'...
remote: Enumerating objects: 7332, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 7332 (delta 186), reused 173 (delta 157), pack-reused 7008 (from 2)
Receiving objects: 100% (7332/7332), 32.12 MiB | 11.15 MiB/s, done.
Resolving deltas: 100% (4351/4351), done.
/content/Ai-Recruiter-Mini-Ai-Service
Branch 'experiment/cross-encoder-v0.6' set up to track remote branch 'experiment/cross-encoder-v0.6' from 'origin'.
Switched to a new branch 'experiment/cross-encoder-v0.6'
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.6 -> FETCH_HEAD
Already up to date.


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.5/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

train       :  9350 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  2000 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  2000 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## Helper Functions

In [5]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Loss function: BoundaryAwareLoss
class BoundaryAwareLoss(torch.nn.Module):
    """MSE + ordinal BCE at boundaries (40/60/75/90)."""
    _BOUNDARIES = [0.40, 0.60, 0.75, 0.90]
    _ALPHA = 0.3

    def forward(self, preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        mse = F.mse_loss(preds, labels)
        ordinal = preds.new_zeros(1)
        for b in self._BOUNDARIES:
            target = (labels >= b).float()
            pred_logit = 20.0 * (preds - b)
            ordinal = ordinal + F.binary_cross_entropy_with_logits(pred_logit, target)
        return mse + self._ALPHA * ordinal / len(self._BOUNDARIES)

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

# Evaluator: CELabelAccEvaluator
class CELabelAccEvaluator:
    def __init__(self, sentence_pairs: list, labels_0_100: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_100 = labels_0_100
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CELabelAccEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_100=[ex.label * 100 for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        arr = np.asarray(preds)
        if arr.ndim == 2:
            pred_classes = arr.argmax(axis=1).tolist()
            true_classes = [_score_to_class_index(t) for t in self.labels_0_100]
            return sum(1 for p, t in zip(pred_classes, true_classes) if p == t) / len(pred_classes)
        pred_100 = [float(p) * 100 for p in preds]
        label_acc = sum(
            1 for p, t in zip(pred_100, self.labels_0_100)
            if abs(p - t) <= 10
        ) / len(pred_100)
        return label_acc

def _score_to_class_index(score: float) -> int:
    """Convert 0-100 score to 5-class index."""
    if score < 40:
        return 0
    elif score < 60:
        return 1
    elif score < 75:
        return 2
    elif score < 90:
        return 3
    else:
        return 4

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("Helper functions loaded.")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Helper functions loaded.


## Load Dataset

In [6]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

Train: 9350 pairs
Val:   2000 pairs
Test:  2000 pairs
Total: 13350 pairs


## Run 1: MSELoss + Spearman, 10 epochs (Baseline)

In [8]:
import torch
from torch.utils.data import DataLoader

run_name = "v0.6-run1-mse-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_examples) // 16 + 1) * 10
warmup_steps = int(total_steps * 0.1)

print(f"🚀 Starting {run_name}...")
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=10,
    loss_fct=torch.nn.MSELoss(),
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run1 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ v0.6-run1-mse-spearman completed.

Validation metrics:
  MAE: 9.5274
  RMSE: 12.4397
  LabelAcc: 0.6285

Test metrics:
  MAE: 9.9416
  RMSE: 13.2407
  LabelAcc: 0.6095


## Run 2: BoundaryAwareLoss + Spearman, 10 epochs

In [9]:
import torch
from torch.utils.data import DataLoader

run_name = "v0.6-run2-boundary-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
boundary_loss = BoundaryAwareLoss()
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_examples) // 16 + 1) * 10
warmup_steps = int(total_steps * 0.1)

print(f"🚀 Starting {run_name}...")
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=10,
    loss_fct=boundary_loss,
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run2 = {
    'run': run_name,
    'loss': 'BoundaryAwareLoss',
    'evaluator': 'Spearman',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

🚀 Starting v0.6-run2-boundary-spearman...


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ v0.6-run2-boundary-spearman completed.

Validation metrics:
  MAE: 10.3511
  RMSE: 13.3883
  LabelAcc: 0.5830

Test metrics:
  MAE: 10.9209
  RMSE: 14.2079
  LabelAcc: 0.5605


## Run 3: MSELoss + LabelAcc, 10 epochs

In [10]:
import torch
from torch.utils.data import DataLoader

run_name = "v0.6-run3-mse-labelacc"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CELabelAccEvaluator.from_input_examples(val_examples, name="val_label_acc")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_examples) // 16 + 1) * 10
warmup_steps = int(total_steps * 0.1)

print(f"🚀 Starting {run_name}...")
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=10,
    loss_fct=torch.nn.MSELoss(),
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run3 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'LabelAcc',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

🚀 Starting v0.6-run3-mse-labelacc...


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ v0.6-run3-mse-labelacc completed.

Validation metrics:
  MAE: 13.9721
  RMSE: 17.6407
  LabelAcc: 0.4325

Test metrics:
  MAE: 14.1458
  RMSE: 17.6712
  LabelAcc: 0.4285


## Summary

In [11]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Run': 'Run 1 (MSE+Spearman)',
        'Val LabelAcc': results_run1['val']['LabelAcc'],
        'Test LabelAcc': results_run1['test']['LabelAcc'],
        'Test MAE': results_run1['test']['MAE'],
        'Test RMSE': results_run1['test']['RMSE']
    },
    {
        'Run': 'Run 2 (Boundary+Spearman)',
        'Val LabelAcc': results_run2['val']['LabelAcc'],
        'Test LabelAcc': results_run2['test']['LabelAcc'],
        'Test MAE': results_run2['test']['MAE'],
        'Test RMSE': results_run2['test']['RMSE']
    },
    {
        'Run': 'Run 3 (MSE+LabelAcc)',
        'Val LabelAcc': results_run3['val']['LabelAcc'],
        'Test LabelAcc': results_run3['test']['LabelAcc'],
        'Test MAE': results_run3['test']['MAE'],
        'Test RMSE': results_run3['test']['RMSE']
    }
])

print("\n📊 Experiment Summary (v0.6 vs v0.2 baseline: 60.76%)\n")
print(summary.to_string(index=False))

best_idx = summary['Test LabelAcc'].idxmax()
best_run = summary.loc[best_idx]
print(f"\n🏆 Best run: {best_run['Run']} with {best_run['Test LabelAcc']:.2%} test LabelAcc")
print(f"  Ceiling broken: {'✅ YES' if best_run['Test LabelAcc'] > 0.6076 else '❌ NO'}")
print(f"  Improvement over v0.2: {(best_run['Test LabelAcc'] - 0.6076) * 100:+.2f} pp")


📊 Experiment Summary (v0.6 vs v0.2 baseline: 60.76%)

                      Run  Val LabelAcc  Test LabelAcc  Test MAE  Test RMSE
     Run 1 (MSE+Spearman)        0.6285         0.6095  9.941631  13.240689
Run 2 (Boundary+Spearman)        0.5830         0.5605 10.920920  14.207909
     Run 3 (MSE+LabelAcc)        0.4325         0.4285 14.145842  17.671208

🏆 Best run: Run 1 (MSE+Spearman) with 60.95% test LabelAcc
  Ceiling broken: ✅ YES
  Improvement over v0.2: +0.19 pp


## Save Reports

In [13]:
import os
import json
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save individual run reports
for run_results in [results_run1, results_run2, results_run3]:
    report = {
        'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
        'dataset_version': 'v0.5',
        'dataset_size': {
            'train': len(train_examples),
            'val': len(val_examples),
            'test': len(test_examples)
        },
        'run': run_results['run'],
        'loss': run_results['loss'],
        'evaluator': run_results['evaluator'],
        'epochs': run_results['epochs'],
        'metrics': {
            'validation': {k: float(v) for k, v in run_results['val'].items()},
            'test': {k: float(v) for k, v in run_results['test'].items()}
        }
    }
    report_path = f"artifacts/reports/{run_results['run']}_report.json"
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"✅ Saved {report_path}")

# Create consolidated report
best_test_labelacc = max(
    results_run1['test']['LabelAcc'],
    results_run2['test']['LabelAcc'],
    results_run3['test']['LabelAcc']
)

consolidated = {
    'experiment': 'cross-encoder-v0.6',
    'dataset': 'v0.5',
    'dataset_size': {
        'train': len(train_examples),
        'val': len(val_examples),
        'test': len(test_examples),
        'total': len(train_examples) + len(val_examples) + len(test_examples)
    },
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'baseline_v0_2': {
        'dataset_size': 7000,
        'test_label_acc': 0.6076,
        'loss': 'MSE',
        'evaluator': 'Spearman',
        'epochs': 10
    },
    'runs': [
        {
            'name': 'Run 1: MSELoss + Spearman',
            'loss': results_run1['loss'],
            'evaluator': results_run1['evaluator'],
            'epochs': results_run1['epochs'],
            'val': {k: float(v) for k, v in results_run1['val'].items()},
            'test': {k: float(v) for k, v in results_run1['test'].items()}
        },
        {
            'name': 'Run 2: BoundaryAwareLoss + Spearman',
            'loss': results_run2['loss'],
            'evaluator': results_run2['evaluator'],
            'epochs': results_run2['epochs'],
            'val': {k: float(v) for k, v in results_run2['val'].items()},
            'test': {k: float(v) for k, v in results_run2['test'].items()}
        },
        {
            'name': 'Run 3: MSELoss + LabelAcc',
            'loss': results_run3['loss'],
            'evaluator': results_run3['evaluator'],
            'epochs': results_run3['epochs'],
            'val': {k: float(v) for k, v in results_run3['val'].items()},
            'test': {k: float(v) for k, v in results_run3['test'].items()}
        }
    ]
}

# Add findings - convert bool to int (0 or 1)
consolidated['findings'] = {
    'best_test_label_acc': float(best_test_labelacc),
    'ceiling_broken': 1 if best_test_labelacc > 0.6076 else 0,
    'improvement_over_v0_2_pp': float((best_test_labelacc - 0.6076) * 100)
}

report_path = 'artifacts/reports/v0.6_complete_report.json'
with open(report_path, 'w') as f:
    json.dump(consolidated, f, indent=2)

print(f"✅ Consolidated report saved to {report_path}")


✅ Saved artifacts/reports/v0.6-run1-mse-spearman_report.json
✅ Saved artifacts/reports/v0.6-run2-boundary-spearman_report.json
✅ Saved artifacts/reports/v0.6-run3-mse-labelacc_report.json
✅ Consolidated report saved to artifacts/reports/v0.6_complete_report.json


## Save to Google Drive (Optional)

In [14]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Find best run by test LabelAcc
best_test_labelacc = max(
    results_run1['test']['LabelAcc'],
    results_run2['test']['LabelAcc'],
    results_run3['test']['LabelAcc']
)

# Save all 3 models
models_to_save = [
    ('artifacts/models/cross-encoder-cv-jd-v0.6-run1-mse-spearman', 'run1-mse-spearman'),
    ('artifacts/models/cross-encoder-cv-jd-v0.6-run2-boundary-spearman', 'run2-boundary-spearman'),
    ('artifacts/models/cross-encoder-cv-jd-v0.6-run3-mse-labelacc', 'run3-mse-labelacc')
]

for src_dir, model_name in models_to_save:
    dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-v0.6-{model_name}"
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(src_dir, dest_dir)
    print(f"✅ Saved: {model_name}")

# Copy all reports
for report_file in os.listdir('artifacts/reports'):
    if report_file.endswith('.json'):
        src = f"artifacts/reports/{report_file}"
        dest = f"{drive_base}/reports/{report_file}"
        shutil.copy(src, dest)
        print(f"✅ Copied report: {report_file}")

# Find and display best model
if results_run1['test']['LabelAcc'] == best_test_labelacc:
    best_run_name = 'Run 1 (MSE + Spearman)'
elif results_run2['test']['LabelAcc'] == best_test_labelacc:
    best_run_name = 'Run 2 (BoundaryAware + Spearman)'
else:
    best_run_name = 'Run 3 (MSE + LabelAcc)'

print(f"\n🏆 Best model: {best_run_name} with {best_test_labelacc:.2%} test LabelAcc")
print(f"\n✅ Done! All 3 models saved to Google Drive!")

Mounted at /content/drive
✅ Saved: run1-mse-spearman
✅ Saved: run2-boundary-spearman
✅ Saved: run3-mse-labelacc
✅ Copied report: v0.6_complete_report.json
✅ Copied report: v0.6-run1-mse-spearman_report.json
✅ Copied report: v0.6-run3-mse-labelacc_report.json
✅ Copied report: v0.6-run2-boundary-spearman_report.json

🏆 Best model: Run 1 (MSE + Spearman) with 60.95% test LabelAcc

✅ Done! All 3 models saved to Google Drive!
